<div align="center">

#### [S. Mussard](https://sites.google.com/view/cv-stphane-mussard/accueil "Homepage") 

</div>

<div align="center">

#### Chapter 12: $(\alpha,\beta)$-Gini Decomposition 

</div>

<div align="center"> 
<a href="https://pytorch.org/"><img src="https://images.squarespace-cdn.com/content/v1/61448910c1763233f3a59e1d/1703953319702-WY9HOFQXJUMV6Y0LXI5C/mouse-pad-white-front-659043916dfe6.jpg?format=1500w" style="max-width: 180px; display: inline" /></a>
</div>

<div align="center">

Cite: S. Mussard (2025), *Machine Learning with Gini Indices: Applications with Python*, Springer.  

</div>

In [ ]:
#!pip install torch
#!pip install jinja2
#!pip install PrettyTable

In [ ]:
#!pip install iteration_utilities

In [ ]:
import pandas as pd
import torch
pd.options.display.float_format = '{:.4f}'.format
import numpy as np
import matplotlib.pyplot as plt
try:
    from OUTLIERS import smirnov_grubbs as grubbs
except ImportError:
    from outliers import smirnov_grubbs as grubbs


## Census Data

In [ ]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo 

census_income = fetch_ucirepo(id=20)  
data = census_income.data.features 

# metadata 
print(census_income.metadata) 

# variables information 
print(census_income.variables) 

## Gini PCA

In [ ]:
# Gini PCA
from Gini_PCA import GiniPca
model = GiniPca(gini_param = 2, n_components = 1)
X = data[['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']]
x = torch.DoubleTensor(X.to_numpy())

In [ ]:
# Circle of correlations
project_var = model.gini_correl_axis(x)
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(1, 1, 1)
for i, j, label in zip(project_var[0,:],project_var[1,:], X.columns):
    plt.text(i, j, label)
    plt.arrow(0,0,i,j,color='gray')
plt.axis((-1.1,1.1,1.1,-1.1))

# Circle
circle = plt.Circle((0,0), radius=1, color='blue', fill=False)
ax.add_patch(circle)
plt.axvline(0, color='r') # vertical axis
plt.axhline(0, color='r') # horizontal axis
plt.xlabel("Axis 1")
plt.ylabel("Axis 2")
plt.show()

In [ ]:
# Compute principal components
components = model.fit(x)[0] 
components

In [ ]:
# Create a DataFrame for the Gini decomposition
components = components.numpy()
df = pd.DataFrame(components[:,0], columns=['component_1'])
df['sex'] = data['sex']
print(df.groupby('sex').describe())

In [ ]:
import seaborn as sns
groups = df.groupby('sex')
plt.figure(figsize=(12, 6))
for name, gender in groups:
    sns.histplot(gender['component_1'], stat='density', kde=True, label=name, element="step", fill=True)
plt.title('Density by Gender')
plt.legend()
plt.show()

## $\alpha$-Gini Decomposition

In [ ]:
# alpha-decomposition
from Gini_decomp import GiniDecomposition
decomposition = GiniDecomposition(alpha=2, method = "absolute")
decomposition.fit(df, 'component_1', 'sex')
decomposition.summary()

In [ ]:
# Contribution of the features
cor_gini = model.gini_correl_axis(x)
cor_gini = cor_gini.numpy()
df_correl = pd.DataFrame(cor_gini, columns=X.columns) 
sum_of_squares = (df_correl ** 2).sum(axis=1)
print((100*df_correl**2).div(sum_of_squares, axis=0))